# Can a Computer Taste? 🍪🤖
**Splash at Yale — E5492**

First, we'll use real age-and-height measurements to see how a computer learns a line. Then **you become the data**: taste four foods and rate them on **sweet**, **salty**, **sour**, and **how enjoyable** they are.

Our question: **if a computer only knows how sweet, salty, and sour a food is, can it predict how much you'll enjoy it?**

👉 Run each cell with **Shift + Enter**. Only change code where it says ✏️.

**Today's path:** height warm-up → taste and explore → guess weights → train → wasabi challenge → learn from some wasabi → test on reserved wasabi → neural network comparison.

## Step 0 — Setup
Load the tools and set the class-data link. Your teacher handles this cell.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression

# Link to the class responses (published Google Sheet as CSV)
TASTE_URL  = "https://docs.google.com/spreadsheets/d/e/2PACX-1vQTBQLZG6f3yxOUXnvs6mx9elfPfl9f2dD_epdzEecIKO-sMcOLiRqfhkKKE7LZJ_XvH4IrCa6X7x0R/pub?output=csv"
WASABI_URL = "https://docs.google.com/spreadsheets/d/e/2PACX-1vTbSHFUShL3bXCPDwzAwKovG9E7asQUY_pFYM6otpUjwESn7SIa63ZgAYixX8UDCZEZ8Fqxa5iGyoqL/pub?output=csv"   # if wasabi is in a separate form, paste its CSV link here; None = look in TASTE_URL

USE_FAKE_DATA = True   # True = practice data (use if the real data isn't ready)

TRAIN_FOODS = ["Pringles", "Pocky", "Sour candy", "Crackers"]
TEST_FOOD   = "Wasabi Peas"
TASTES      = ["sweet", "salty", "sour"]   # what the computer gets to see
TARGET      = "enjoyable"                  # what it tries to predict

## Warm-up — Can a computer learn a line? 📏 (about 7 minutes)
Start with the female group: **age is the input**, and **height is the output**. Imagine drawing a line through the middle of this cloud of points.

🤔 Where would your line go? Would it hit every dot? Does the pattern look perfectly straight, or does it bend?

These are real measurements of people **ages 6–15**. We use the dataset's female and male groups, fitting a separate line to each. These are different people measured once, not one person's growth over time.

**Source:** CDC NHANES 2017–2018, [standing height (BMXHT, cm)](https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/BMX_J.htm) and [age and group (RIDAGEYR, RIAGENDR)](https://wwwn.cdc.gov/nchs/data/nhanes/public/2017/datafiles/DEMO_J.htm), joined by participant ID. This embedded teaching sample has 8 people per age per group (160 total), sampled with seed 42. It is unweighted: the lines describe this sample, not national growth standards.

In [ ]:
# Real CDC NHANES measurements, embedded so this demo works offline.
# Each row holds eight people of one age: ages 6 through 15, in order.
height_samples = {
    "Female": [
        [120.7, 121.5, 113.6, 121.7, 122.5, 123.5, 129.3, 130.3],  # age 6
        [133.9, 133.4, 123.9, 128.4, 129.9, 123.5, 131.4, 113.2],  # age 7
        [136.2, 129.6, 138.2, 131.7, 150.0, 123.1, 127.0, 129.6],  # age 8
        [142.1, 133.5, 126.7, 138.3, 143.2, 142.5, 152.2, 139.6],  # age 9
        [147.6, 143.1, 140.7, 131.8, 153.4, 148.2, 142.2, 156.8],  # age 10
        [158.8, 165.1, 159.5, 167.2, 158.7, 163.6, 151.6, 162.7],  # age 11
        [152.6, 152.1, 168.8, 156.6, 160.0, 164.0, 155.7, 148.4],  # age 12
        [157.8, 163.9, 156.8, 165.1, 165.3, 151.1, 151.0, 146.0],  # age 13
        [157.8, 158.7, 157.9, 156.6, 165.3, 160.3, 157.6, 159.2],  # age 14
        [170.0, 176.9, 156.1, 167.3, 170.9, 173.1, 166.5, 155.6],  # age 15
    ],
    "Male": [
        [124.7, 124.1, 128.5, 109.7, 117.7, 125.9, 121.7, 129.2],  # age 6
        [118.7, 128.6, 128.0, 133.5, 123.1, 128.8, 112.2, 125.0],  # age 7
        [136.8, 133.8, 133.4, 122.9, 134.2, 120.0, 137.1, 129.8],  # age 8
        [134.0, 137.1, 131.7, 127.8, 149.9, 132.4, 130.8, 138.0],  # age 9
        [154.1, 138.7, 149.0, 144.9, 142.4, 146.8, 137.4, 131.1],  # age 10
        [148.6, 142.7, 140.4, 153.5, 150.4, 162.1, 148.5, 165.6],  # age 11
        [153.7, 166.4, 161.9, 157.3, 159.3, 150.5, 168.7, 154.3],  # age 12
        [181.6, 172.4, 172.3, 169.2, 167.4, 143.3, 152.0, 160.3],  # age 13
        [174.5, 165.5, 163.4, 181.5, 155.2, 174.5, 174.8, 161.0],  # age 14
        [185.5, 157.0, 184.5, 172.6, 181.0, 170.3, 174.7, 175.7],  # age 15
    ],
}
height_rows = []
for group, heights_by_age in height_samples.items():
    for age, heights in zip(range(6, 16), heights_by_age):
        for height in heights:
            height_rows.append({"group": group, "age": age, "height_cm": height})
height_data = pd.DataFrame(height_rows)

female = height_data[height_data["group"] == "Female"]
plt.figure(figsize=(8, 4))
plt.scatter(female["age"], female["height_cm"], s=36, alpha=0.65, color="#0072B2")
plt.xlabel("Age (years)")
plt.ylabel("Height (cm)")
plt.xticks(range(6, 16))
plt.title("Girls / young women: can you see a pattern?")
plt.grid(alpha=0.15)
plt.tight_layout()
plt.show()
print("Each dot is a different person. People of the same age can have different heights.")

### From dots to a rule
A straight line is a small prediction machine:

$$\widehat{\text{height}} = b + w \times \text{age}$$

The hat means **predicted**. The **weight** $w$ tilts the line: one extra year adds $w$ cm to its prediction. The **start** $b$ moves the line up or down; it is not a useful birth-height estimate when our data starts at age 6.

For example, the made-up rule $\widehat{\text{height}} = 90 + 5 \times \text{age}$ predicts $90 + 5 \times 10 = 140$ cm at age 10. If a person's measured height is 147 cm, that prediction misses by $|140 - 147| = 7$ cm.

**Training** means using the dots to choose the line's numbers. Run the cell below: `.fit()` learns $b$ and $w$ for each group. The dashed vertical segment shows one prediction's miss.

We describe accuracy using **mean absolute error**: add the sizes of all the misses and divide by the number of people. Lower is better.

In [ ]:
height_models = {}
height_ages = pd.DataFrame({"age": np.linspace(6, 15, 100)})
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharex=True, sharey=True)
for ax, group, color in zip(axes, ["Female", "Male"], ["#0072B2", "#D55E00"]):
    people = height_data[height_data["group"] == group]
    fitted_line = LinearRegression().fit(people[["age"]], people["height_cm"])
    height_models[group] = fitted_line
    fitted_heights = fitted_line.predict(people[["age"]])
    height_error = np.abs(fitted_heights - people["height_cm"]).mean()

    ax.scatter(people["age"], people["height_cm"], color=color, s=22, alpha=0.35)
    ax.plot(height_ages["age"], fitted_line.predict(height_ages), color=color, lw=3, label="Learned line")
    example = people[people["age"] == 10].iloc[[0]]
    predicted_height = fitted_line.predict(example[["age"]])[0]
    actual_height = example["height_cm"].iloc[0]
    ax.plot([10, 10], [predicted_height, actual_height], "k--", lw=2, label="One prediction's miss")
    ax.set(title=f"{group} group", xlabel="Age (years)", xlim=(5.5, 15.5), ylim=(100, 195))
    ax.set_xticks([6, 8, 10, 12, 14, 15])
    ax.grid(alpha=0.15)
    ax.legend(fontsize=9)
    print(f"{group}: predicted height = {fitted_line.intercept_:.1f} + {fitted_line.coef_[0]:.1f} × age (cm)")
    print(f"  Off by {height_error:.1f} cm on average on these training examples.")
axes[0].set_ylabel("Height (cm)")
plt.tight_layout()
plt.show()

### ✏️ Try an age
Change only `prediction_age` below to a number from **6 to 15**. The computer plugs your age into each learned line and shows the predicted height in **feet and inches**, rounded to the nearest inch. The calculation underneath stays in centimeters.

This predicts a height from each group's pattern, not your exact height or how tall you will become. People vary, and growth is not a straight line across a whole lifetime. We keep predictions inside the ages we studied.

In [ ]:
prediction_age = 15   # ✏️ Try an age from 6 to 15.

if 6 <= prediction_age <= 15:
    age_input = pd.DataFrame({"age": [prediction_age]})
    for group, fitted_line in height_models.items():
        height_prediction = fitted_line.predict(age_input)[0]
        total_inches = round(height_prediction / 2.54)  # 1 inch = 2.54 cm
        feet, inches = divmod(total_inches, 12)
        print(f"📏 {group} group, age {prediction_age:g}: predicted height about {feet} ft {inches} in")
        print(f"   {fitted_line.intercept_:.1f} + {fitted_line.coef_[0]:.1f} × {prediction_age:g} ≈ {height_prediction:.1f} cm")
else:
    print("Choose an age from 6 to 15: we don't have examples outside that range.")

### Now use the same idea for food 🍪
The height model used **one input** (age) to predict **one output** (height). Our food model will use **three inputs** (sweet, salty, sour) to predict **one output** (enjoyable).

It will have a start and three weights. This time, **you choose the weights first**, then let the computer learn them.

## Step 1 — Load the data
Each row of the spreadsheet is one person. We reshape it so **each row is one person tasting one food**.

In [ ]:
def to_long(wide):
    """Turn 'Food | rating' columns into rows: student, food, sweet, salty, sour, enjoyable."""
    wide = wide.copy()
    wide.columns = [str(c).strip() for c in wide.columns]
    wide["student"] = range(1, len(wide) + 1)
    rows = []
    for col in wide.columns:
        if "|" not in col:
            continue
        food, rating = [p.strip() for p in col.split("|", 1)]
        part = pd.DataFrame({"student": wide["student"], "food": food,
                             "rating": rating.lower(),
                             "value": pd.to_numeric(wide[col], errors="coerce")})
        rows.append(part)
    if not rows:
        return pd.DataFrame(columns=["student", "food"] + TASTES + [TARGET])
    long = pd.concat(rows).pivot_table(index=["student", "food"], columns="rating",
                                       values="value").reset_index()
    long.columns.name = None
    long = long.reindex(columns=["student", "food"] + TASTES + [TARGET])
    return long.dropna(subset=TASTES + [TARGET])

def fake_data(n=20, seed=0):
    """Made-up ratings that look roughly like a real class."""
    rng = np.random.default_rng(seed)
    #               sweet salty sour  enjoyable
    profile = {"Pringles":    (2, 8, 1, 8), "Pocky":    (8, 2, 1, 8),
               "Sour candy":  (7, 1, 8, 7), "Crackers": (7, 3, 1, 6),
               "Wasabi Peas": (1, 7, 1, 4)}
    rows = []
    for s in range(1, n + 1):
        for food, (sw, sa, so, en) in profile.items():
            r = lambda m: int(np.clip(round(m + rng.normal(0, 1.5)), 1, 10))
            rows.append({"student": s, "food": food, "sweet": r(sw), "salty": r(sa),
                         "sour": r(so), "enjoyable": r(en + rng.normal(0, 1.5))})
    return pd.DataFrame(rows)

data = None
if not USE_FAKE_DATA:
    try:
        data = to_long(pd.read_csv(TASTE_URL))
        if (data["food"].isin(TRAIN_FOODS)).sum() == 0:
            print("⚠️  No responses yet — using practice data for now.")
            data = None
    except Exception as e:
        print("⚠️  Couldn't load the class data (", type(e).__name__, ") — using practice data.")
        data = None
if data is None:
    data = fake_data()
    print("📋 Using PRACTICE data.")
else:
    print("✅ Loaded REAL class data!")
    if WASABI_URL:
        try:
            extra = to_long(pd.read_csv(WASABI_URL))
            extra = extra[extra["food"] == TEST_FOOD]
            if len(extra):
                data = pd.concat([data[data["food"] != TEST_FOOD], extra], ignore_index=True)
        except Exception as e:
            print(f"⚠️ Couldn't load the separate wasabi sheet ({type(e).__name__}); keeping the class data.")

train  = data[data["food"].isin(TRAIN_FOODS)].reset_index(drop=True)
wasabi = data[data["food"] == TEST_FOOD].reset_index(drop=True)
print(f"{train['student'].nunique()} tasters, {len(train)} training rows, {len(wasabi)} wasabi rows")
train.head(8)

## Step 2 — Look at the data
What does the class think, on average?

These averages help us see patterns. **We train on each person's ratings**, not just these four averages: different people can enjoy the same food differently.

In [ ]:
train.groupby("food")[TASTES + [TARGET]].mean().round(1)

### How to read the taste plots
Read one panel at a time: **farther right** means more of that taste; **higher up** means more enjoyable.

**Small, faint dots** are individual ratings. **Large, outlined symbols** mark each food's class average; match their color and shape to the legend. Tiny, separate nudges in each direction uncover overlapping dots without changing the data used for training.

🤔 Find Pringles in the saltiness panel. Is the average high or low? How much do individual opinions spread out?

In [ ]:
plot_rng = np.random.default_rng(7)  # Same picture each time you run it.
food_colors = ["#0072B2", "#D55E00", "#009E73", "#CC79A7"]
food_markers = ["o", "s", "^", "D"]
fig, axes = plt.subplots(1, len(TASTES), figsize=(13, 4.5), sharey=True, squeeze=False)
for ax, taste in zip(axes[0], TASTES):
    for food, color, marker in zip(TRAIN_FOODS, food_colors, food_markers):
        d = train[train["food"] == food]
        if d.empty:
            continue
        x = d[taste] + plot_rng.uniform(-0.16, 0.16, len(d))
        y = d[TARGET] + plot_rng.uniform(-0.16, 0.16, len(d))
        ax.scatter(x, y, color=color, marker=marker, s=22, alpha=0.25, edgecolors="none")
        ax.scatter(d[taste].mean(), d[TARGET].mean(), color=color, marker=marker,
                   s=150, edgecolors="#222222", linewidths=1.2, label=food, zorder=3)
    ax.set(title=f"{taste.capitalize()} → enjoyment?", xlabel=f"How {taste} (1–10)",
           xlim=(0.5, 10.5), ylim=(0.5, 10.5), xticks=[1, 3, 5, 7, 9, 10])
    ax.set_yticks([1, 3, 5, 7, 9, 10])
    ax.grid(alpha=0.15)
    ax.set_axisbelow(True)
axes[0, 0].set_ylabel("How enjoyable (1–10)")
handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", bbox_to_anchor=(0.5, 0.94), ncol=4, frameon=False)
fig.suptitle("Small dots = individual ratings     Large symbols = food averages", y=1.0)
plt.tight_layout(rect=[0, 0, 1, 0.86])
plt.show()

🤔 **Talk with your partner:** Does saltier = more enjoyable? Sweeter? More sour? Do people agree with each other?

## Step 3 — You vs. the computer: guess the weights
**Why do this?** To see what training changes. You and the computer use **the same inputs and the same prediction rule**. First you choose its numbers by hand; then the computer chooses them using the data.

Our model is a recipe:

$$\widehat{\text{enjoyment}} = \text{start} + w_{\text{sweet}}\cdot\text{sweet} + w_{\text{salty}}\cdot\text{salty} + w_{\text{sour}}\cdot\text{sour}$$

A **weight** controls the prediction. Holding the other inputs fixed, a positive sweet weight makes the prediction rise when sweetness rises; a negative weight makes it fall; zero means this model ignores sweetness. It does not prove that sweetness causes enjoyment.

For example, with the guesses below, sweet = 8, salty = 2, sour = 1 gives $3 + 0.3(8) + 0.3(2) + 0(1) = 6$. If the person actually rated enjoyment 8, the miss is $|6 - 8| = 2$ points.

Our score is **mean absolute error**: the average size of those misses, in rating points. Lower is better.

✏️ **Try two sets of weights with your partner.** Record each score, then leave your better guess in the cell and run it again. You are doing a small search for weights with low error!

In [ ]:
# ✏️ YOUR GUESSES
start    = 3.0
w_sweet  = 0.3
w_salty  = 0.3
w_sour   = 0.0

guess = start + w_sweet*train["sweet"] + w_salty*train["salty"] + w_sour*train["sour"]
my_error = (guess - train[TARGET]).abs().mean()
print(f"Your predictions are off by {my_error:.2f} points on average (out of 10).")

## Step 4 — Let the computer train
**Training** means choosing weights from examples to make a chosen error rule as small as possible. You searched by hand; `.fit()` calculates weights from all the training rows. The computer still uses the same recipe as you.

Compare your average misses below. **The point is what changed: the weights, not the inputs or the prediction rule.** Neither model has seen wasabi as a training example.

**A detail about our score:** [scikit-learn's `LinearRegression`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html) fits by least squares, a rule that gives large misses extra importance. We report **mean absolute error** because it is easy to read. The computer is not guaranteed to beat your guess on this score.

These scores use the rows we trained on. A low training error does **not** guarantee success on a new food.

In [ ]:
model = LinearRegression()
model.fit(train[TASTES], train[TARGET])

computer_error = np.abs(model.predict(train[TASTES]) - train[TARGET]).mean()
print(f"Computer's start: {model.intercept_:.2f}")
for taste, w in zip(TASTES, model.coef_):
    print(f"  {taste}: each extra rating point changes predicted enjoyment by {w:+.2f} points")
print(f"\n🤖 Computer: off by {computer_error:.2f} points on average.")
print(f"✏️ Your guess: off by {my_error:.2f} points on average.")
if np.isclose(my_error, computer_error, atol=0.005, rtol=0):
    print("🤝 A tie to about one hundredth of a point!")
elif my_error < computer_error:
    print("🏆 Your weights win on average absolute miss this round.")
else:
    print("🤖 The computer's weights win on average absolute miss this round.")

fig, ax = plt.subplots(figsize=(7, 2.6))
bars = ax.barh(["Your guessed weights", "Computer's fitted weights"],
               [my_error, computer_error], color=["#D55E00", "#0072B2"])
ax.bar_label(bars, fmt="%.2f", padding=5)
ax.set_xlim(0, max(my_error, computer_error, 0.1) * 1.25)
ax.set_xlabel("Average miss (rating points) — lower is better")
ax.set_title("Same prediction rule + same rows; different weights")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## Step 5 — The challenge: wasabi peas 🟢
✍️ **Before running:** write down your guess — how enjoyable will the class find wasabi peas?

The computer only sees how **sweet, salty, and sour** the wasabi peas are. Let's see what it predicts… It might do well or poorly; we need to check.

We keep wasabi predictions between 1 and 10 so they stay on our rating scale.

In [ ]:
if len(wasabi) == 0:
    print("No complete wasabi ratings yet. A teacher/pilot taster can submit one response first.")
    print("Then re-run Step 1 and this prediction cell; save the reveal until after tasting.")
else:
    wasabi = wasabi.copy()
    wasabi["predicted"] = model.predict(wasabi[TASTES]).clip(1, 10)
    print("Average wasabi taste ratings (enjoyment is hidden for now):")
    print(wasabi[TASTES].mean().round(1).to_string())
    print(f"\n🤖 Computer predicts enjoyment: {wasabi['predicted'].mean():.1f} / 10")

👅 **Now everyone eats a wasabi pea and submits their ratings.** Then run the reveal below.

**For fresh responses:** re-run Step 1's loading cell first. The reveal uses the loaded wasabi rows and recomputes their predictions, so refreshing won't break it. More tasters may change the predicted average too. If the four training foods also gained responses, re-run Steps 3 and 4 before the reveal.

In [ ]:
if len(wasabi) == 0:
    print("No complete wasabi ratings to reveal yet. Submit them, then re-run Step 1 and this cell.")
else:
    wasabi_predictions = model.predict(wasabi[TASTES]).clip(1, 10)
    actual, predicted = wasabi[TARGET].mean(), wasabi_predictions.mean()
    wasabi_error = np.abs(wasabi_predictions - wasabi[TARGET]).mean()
    print(f"🤖 Predicted class average: {predicted:.1f}    😋 Actual class average: {actual:.1f}")
    print(f"Gap between class averages: {abs(actual - predicted):.1f} points.")
    print(f"Individual predictions are off by {wasabi_error:.1f} points on average.")

    train_avg = train.groupby("food")[TARGET].mean()
    names = list(train_avg.index) + ["Wasabi\n(predicted)", "Wasabi\n(actual)"]
    vals = list(train_avg.values) + [predicted, actual]
    colors = ["#9aa5b1"] * len(train_avg) + ["#4c78a8", "#e45756"]
    fig, ax = plt.subplots(figsize=(8, 4))
    bars = ax.bar(names, vals, color=colors)
    ax.bar_label(bars, fmt="%.1f", padding=3)
    ax.set(ylim=(0, 11), ylabel="Average enjoyment (1–10)",
           title="How did the computer do on a food it never trained on?")
    plt.tight_layout()
    plt.show()

## Step 6 — Does seeing wasabi examples help? 🔀
The first model learned from four foods and then faced wasabi. Now change **the training examples**, while keeping the same sweet, salty, and sour inputs.

| Model | Examples used for training | Examples used for testing |
|---|---|---|
| A: no wasabi examples | All ratings of the original four foods | Reserved wasabi responses |
| B: with wasabi examples | Those same ratings, plus some wasabi responses | The **same** reserved wasabi responses |

Shuffle the wasabi responses and reserve about **20% for testing** before fitting either model. With 20 complete wasabi responses, 16 join Model B's training data and 4 stay out of **both** models.

**Why shuffle?** To avoid choosing the test group just by submission order. The seed fixes one random split so we can reproduce it. Shuffling alone does not improve a linear model: the experiment is about **adding relevant training examples**. Do not keep changing the split until the result looks better.

This tests new **wasabi responses**, not necessarily entirely new people: their other-food ratings may already be in training. Row numbers from separate forms are not matched identities. Neither model uses food names or student IDs as prediction inputs.

We saw a class summary earlier, but these fresh models never fit on the reserved answers. For a real final evaluation, reserve examples before examining their answers. Improvement is possible, not guaranteed.

In [ ]:
# Keep the reserved wasabi answers out of both models.
wasabi_ready = len(wasabi) >= 2
wasabi_train = wasabi.iloc[:0].copy()
wasabi_test = wasabi.iloc[:0].copy()
mixed_train = train.copy()
model_without_wasabi = LinearRegression().fit(train[TASTES], train[TARGET])
model_with_wasabi = None
wasabi_models = {}
if not wasabi_ready:
    print("We need at least 2 complete wasabi responses: one to learn from and one to test.")
    print("Collect more responses, then rerun Step 1 and this section. A full class gives a better test.")
else:
    shuffled_wasabi = wasabi.sample(frac=1, random_state=42)
    n_wasabi_test = max(1, int(np.ceil(0.2 * len(shuffled_wasabi))))
    wasabi_test = shuffled_wasabi.iloc[:n_wasabi_test].copy()
    wasabi_train = shuffled_wasabi.iloc[n_wasabi_test:].copy()
    mixed_train = pd.concat([train, wasabi_train], ignore_index=True)
    model_with_wasabi = LinearRegression().fit(mixed_train[TASTES], mixed_train[TARGET])
    wasabi_models = {"A: no wasabi examples": model_without_wasabi,
                     "B: with wasabi examples": model_with_wasabi}
    print(f"A trains on {len(train)} original-food ratings.")
    print(f"B trains on those same ratings + {len(wasabi_train)} wasabi ratings.")
    print(f"🔒 Both are tested on the SAME {len(wasabi_test)} reserved wasabi ratings.")
    if len(wasabi_test) < 5:
        print("This is a small test group: a few people's opinions can change the result.")

### Two predictions for the same reserved wasabi response
Both models receive exactly the same sweet, salty, and sour inputs. Only B has learned from other wasabi examples. Neither receives this response's enjoyment rating.

🤔 Which prediction will be closer? We limit both predictions to the 1–10 rating scale.

In [ ]:
if wasabi_ready:
    wasabi_example = wasabi_test.iloc[[0]]
    print("Reserved wasabi taste ratings:")
    print(wasabi_example[TASTES].iloc[0].to_string())
    for name, fitted_model in wasabi_models.items():
        prediction = fitted_model.predict(wasabi_example[TASTES]).clip(1, 10)[0]
        print(f"🤖 {name}: predicted enjoyment {prediction:.1f} / 10")
    print("Run the next cell to reveal the actual enjoyment.")
else:
    print("Waiting for at least 2 complete wasabi responses before making this comparison.")

### Reveal and compare every reserved wasabi response
One lucky prediction is not enough. Compute each model's **mean absolute error on the same entire wasabi test group**. A lower score means a smaller average miss. This score uses individual ratings, not just the gap between class averages.

In [ ]:
wasabi_errors = {}
if wasabi_ready:
    print(f"😋 Actual enjoyment for that response: {wasabi_example[TARGET].iloc[0]:.1f} / 10")
    for name, fitted_model in wasabi_models.items():
        predictions = fitted_model.predict(wasabi_test[TASTES]).clip(1, 10)
        error = np.abs(predictions - wasabi_test[TARGET]).mean()
        wasabi_errors[name] = error
        print(f"{name}: off by {error:.2f} points on average on reserved wasabi ratings.")
    improvement = wasabi_errors["A: no wasabi examples"] - wasabi_errors["B: with wasabi examples"]
    if np.isclose(improvement, 0, atol=0.005, rtol=0):
        print("🤝 About the same average miss in this split.")
    elif improvement > 0:
        print(f"✅ Adding wasabi examples reduced the average miss by {improvement:.2f} points in this split.")
    else:
        print(f"Adding wasabi examples increased the average miss by {-improvement:.2f} points in this split.")
    fig, ax = plt.subplots(figsize=(8, 3.5))
    bars = ax.bar(list(wasabi_errors), list(wasabi_errors.values()), color=["#0072B2", "#009E73"])
    ax.bar_label(bars, fmt="%.2f", padding=4)
    ax.set_ylim(0, max(max(wasabi_errors.values()), 0.1) * 1.3)
    ax.set_ylabel("Average miss on reserved wasabi ratings")
    ax.set_title("Does seeing other wasabi examples help? Lower is better")
    plt.tight_layout()
    plt.show()
else:
    print("No fair before/after comparison yet: we need separate wasabi training and test responses.")

🤔 **Discuss:** Did wasabi examples help in this split? Why would testing on the same responses used for training give a misleading impression?

**The idea:** training examples closer to what we want to predict can help. They don't guarantee a large improvement: people disagree, inputs may not describe every preference, and a small test set is noisy. If two foods have the same three taste ratings, these models cannot distinguish them by name.

Next, we keep the mixed training data and reserved wasabi test set fixed, and change the model to a neural network.

## Step 7 — Would a neural network do better? 🧠 (about 6 minutes)
We have been using **linear regression**. Now we'll train a small **neural network** on the same individual ratings, using the same three inputs and the same output.

**Think of a neuron as a tiny calculator, not a thinking brain cell.** It multiplies inputs by weights, adds a starting value, and applies a rule. Our hidden neurons use this rule:

$$h = \max(0,\ b + w_1x_1 + w_2x_2 + w_3x_3)$$

That means **keep a positive result; replace a negative result with zero**. This rule is called ReLU. Combining these calculators lets the network learn patterns that bend: for example, sweetness could matter differently at different saltiness levels.

Our network has **3 inputs → 6 hidden neurons → 1 predicted enjoyment**. “Hidden” just means an intermediate calculation, rather than an input or the final answer. The output combines the six results with another weighted sum.

🤔 Before training: will a more complicated model necessarily make better predictions?

In [ ]:
# An example network with six hidden neurons. No edits needed.
fig, ax = plt.subplots(figsize=(9, 4))
input_y = np.linspace(0.25, 0.75, len(TASTES))
hidden_y = np.linspace(0.08, 0.92, 6)
for y in input_y:
    for h in hidden_y:
        ax.plot([0, 1], [y, h], color="#b8c2cc", lw=0.8, zorder=1)
for h in hidden_y:
    ax.plot([1, 2], [h, 0.5], color="#b8c2cc", lw=0.8, zorder=1)
ax.scatter(np.zeros(len(TASTES)), input_y, s=600, color="#0072B2", zorder=2)
ax.scatter(np.ones(6), hidden_y, s=600, color="#009E73", zorder=2)
ax.scatter([2], [0.5], s=750, color="#D55E00", zorder=2)
for y, taste in zip(input_y, TASTES):
    ax.text(-0.12, y, taste, ha="right", va="center", fontsize=11)
for i, h in enumerate(hidden_y, start=1):
    ax.text(1, h, str(i), ha="center", va="center", color="white")
ax.text(2.12, 0.5, "predicted\nenjoyment", va="center", fontsize=11)
for x, label in [(0, "Taste inputs"), (1, "Hidden calculators"), (2, "Output")]:
    ax.text(x, 1.08, label, ha="center", fontsize=12)
ax.set(xlim=(-0.55, 2.65), ylim=(-0.05, 1.2))
ax.axis("off")
plt.title("Information flows left to right; each connection has a weight", pad=12)
plt.tight_layout()
plt.show()

### ✏️ Your turn: build the network (3 minutes)
Work with your partner. Replace the two `None` placeholders in the next cell:

1. Set `hidden_neurons` to **6**, matching the diagram. This chooses how many hidden calculators your network has.
2. Set `prediction_column` to **TARGET** (without quotes). This tells the network to learn to predict enjoyment.

**Hint:** `TASTES` names the input columns; `TARGET` names the answer column. The code below uses your choices to build and train the network.

`None` means “not filled in yet.” If you run the notebook before completing the exercise, the provided training code uses six neurons and enjoyment as defaults, so class can continue.

**Finished early?** Change only the neuron count to **3 or 12**, rerun your cell, the training cell, and the comparison cell. Does a larger network help? The diagram is the six-neuron example. Keep the data split fixed. If you use test scores to choose a network, you need fresh responses for a final test.

In [ ]:
# ✏️ YOUR TWO LINES — replace each None.
hidden_neurons = None      # Number of hidden neurons: start with 6.
prediction_column = None   # Which column do we predict? Use TARGET.

### Train it, then compare
With six hidden neurons, the network repeatedly adjusts **31 learned numbers**, compared with **4** for linear regression. More adjustable numbers can capture complicated patterns, but can also fit quirks of a small dataset. A good fit to training examples with poor predictions on new examples is **overfitting**.

We use scikit-learn's [`MLPRegressor`](https://scikit-learn.org/stable/modules/generated/sklearn.neural_network.MLPRegressor.html), already available in Colab. Its input scaling is learned from training rows only.

Train the neural network on **exactly Model B's mixed training rows**, then test it on **the same reserved wasabi responses**. Now the model type changes while the data stays fixed. Training scores use raw predictions; reserved wasabi scores limit both models' predictions to 1–10. All displayed errors are mean absolute errors.

With fewer than 2 wasabi responses, we demonstrate fitting on the original foods only and skip the wasabi comparison. After new responses arrive, rerun Steps 1, 3, 4, 6, and this section.

In [ ]:
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Provided support code: use the exercise choices, or runnable defaults.
chosen_neurons = 6 if hidden_neurons is None else hidden_neurons
chosen_target = TARGET if prediction_column is None else prediction_column
if chosen_neurons not in (3, 6, 12):
    print("Choose 3, 6, or 12 neurons in your cell. Using 6 for now.")
    chosen_neurons = 6
if chosen_target != TARGET:
    print("Use TARGET without quotes to predict enjoyment. Using enjoyment for now.")
    chosen_target = TARGET
if hidden_neurons is None or prediction_column is None:
    print("✏️ Using defaults for unfinished choices. Complete your two lines above!")

# Your choices determine the network size and the answers it learns from.
neural_model = make_pipeline(
    StandardScaler(),
    MLPRegressor(hidden_layer_sizes=(chosen_neurons,), activation="relu", solver="lbfgs",
                 alpha=1.0, max_iter=3000, max_fun=30000, random_state=42)
)
nn_training_rows = mixed_train
neural_model.fit(nn_training_rows[TASTES], nn_training_rows[chosen_target])
network = neural_model.named_steps["mlpregressor"]
learned_numbers = sum(a.size for a in network.coefs_ + network.intercepts_)
print(f"🧠 Network trained on {len(nn_training_rows)} individual ratings; learned {learned_numbers} numbers.")
print(f"Your network: {chosen_neurons} hidden neurons; predicting {chosen_target}.")
if train["student"].nunique() < 5:
    print("📋 Only a few tasters so far: this tests the code, not which model is generally better.")

In [ ]:
comparison_linear = model_with_wasabi if wasabi_ready else model_without_wasabi
comparison_models = {"Linear regression": comparison_linear, "Neural network": neural_model}
evaluation_sets = [("Training ratings (seen)", nn_training_rows)]
if wasabi_ready:
    evaluation_sets.append(("Reserved wasabi (unseen ratings)", wasabi_test))
else:
    print("No separate wasabi test set yet. We can compare training fit only.")

fig, axes = plt.subplots(1, len(evaluation_sets), figsize=(5 * len(evaluation_sets), 3.5), squeeze=False)
neural_comparison = []
for ax, (label, rows) in zip(axes[0], evaluation_sets):
    errors = []
    for name, fitted_model in comparison_models.items():
        predictions = fitted_model.predict(rows[TASTES])
        if label == "Reserved wasabi (unseen ratings)":
            predictions = predictions.clip(1, 10)
        error = np.abs(predictions - rows[TARGET]).mean()
        errors.append(error)
        neural_comparison.append({"examples": label, "model": name, "average miss": error})
        print(f"{label} — {name}: off by {error:.2f} points on average.")
    bars = ax.bar(list(comparison_models), errors, color=["#0072B2", "#009E73"])
    ax.bar_label(bars, fmt="%.2f", padding=4)
    ax.set_title(label)
    ax.set_ylabel("Average miss (rating points)")
# Use the same scale so the panels can be compared fairly.
error_limit = max(0.1, max(row["average miss"] for row in neural_comparison)) * 1.3
for ax in axes[0]:
    ax.set_ylim(0, error_limit)
fig.suptitle("Lower is better — training fit is not the same as performance on unseen examples")
plt.tight_layout()
plt.show()

🤔 **Talk with your partner:** Which model fits the training examples better? Does it also predict the reserved wasabi ratings better?

The first comparison changed **the training examples**. This comparison changes **the model**, keeping the data fixed. Neither improvement is guaranteed.

A small test group is not enough to declare an overall winner. If we use these results to choose or adjust a model, we need fresh responses for a final test. More complex models generally need more data.

## 🧠 Discussion
1. What did the model get right? What did it get wrong?
2. Is "how sweet" easier to predict than "how enjoyable"? Why might people agree on one more than the other?
3. What affects enjoyment that our three taste ratings might not capture? (Hint: pinch your nose 👃)
4. Why reserve some wasabi responses instead of training on all of them?
5. **Can a computer taste?**

### 🚀 Bonus (if you finish early)
✏️ Try fewer inputs: change `bonus_tastes` below to `["salty"]`, `["sweet"]`, or two of our tastes. Run this cell to compare its average miss with the original three-input computer model.

Which choice works best on these training rows? That alone doesn't prove which taste causes enjoyment or which choice will work best on a new food.

In [ ]:
bonus_tastes = ["salty"]   # ✏️ Choose from "sweet", "salty", "sour".

if not bonus_tastes or any(taste not in TASTES for taste in bonus_tastes):
    print("Choose at least one of: sweet, salty, sour.")
else:
    bonus_model = LinearRegression().fit(train[bonus_tastes], train[TARGET])
    bonus_error = np.abs(bonus_model.predict(train[bonus_tastes]) - train[TARGET]).mean()
    print(f"Using {', '.join(bonus_tastes)}: off by {bonus_error:.2f} points on average.")
    print(f"Using all three tastes: off by {computer_error:.2f} points on average.")